In [1]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from datasets import load_from_disk

c:\Users\ASUS\Desktop\sentiment-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Dataset

In [2]:
DATA_PATH = Path ("../data/tokenized")

dataset = load_from_disk (DATA_PATH)
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'target', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1536
    })
    validation: Dataset({
        features: ['sentence', 'label', 'target', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 384
    })
    test: Dataset({
        features: ['sentence', 'label', 'target', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 339
    })
})

In [3]:
dataset["train"].column_names

['sentence',
 'label',
 'target',
 'input_ids',
 'token_type_ids',
 'attention_mask']

### Rename and Remove Columns

In [4]:
dataset = dataset.rename_column ("target", "labels")

In [6]:
dataset = dataset.remove_columns (["sentence", "label"])

In [7]:
dataset["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

### Convert to Torch

In [8]:
dataset.set_format (
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "labels"
    ]
    )

In [9]:
dataset["train"][0]

{'labels': tensor(2),
 'input_ids': tensor([  101,  1050,  1009,  1015,  2177,  2097,  3477,  7327,  2099, 16048,
          1012,  1019,  1049,  1997,  1996, 12598,  3976,  2588,  5494,  1010,
          1998,  1996,  3588,  7680,  1999,  2262,  1012,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 

### Build DataLoaders

In [11]:
train_loader = DataLoader (dataset["train"], batch_size=16, shuffle=True)
val_loader = DataLoader (dataset["validation"], batch_size=16, shuffle=False)
test_loader = DataLoader (dataset["test"], batch_size=16, shuffle=False)

In [12]:
batch = next(iter(train_loader))
batch.keys()

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])

In [13]:
for key, value in batch.items():
    print (key, value.shape)

labels torch.Size([16])
input_ids torch.Size([16, 64])
token_type_ids torch.Size([16, 64])
attention_mask torch.Size([16, 64])


In [14]:
batch["labels"]

tensor([2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1])

In [15]:
batch["input_ids"][0]

tensor([  101,  1996,  4769,  3295,  2003,  3024,  2000,  1037,  2334, 11486,
         1010,  9544,  8231,  2011,  9949,  6726,  1010,  2029,  2059,  3594,
         1996,  4769,  3295,  2000,  3229,  1996, 10439, 15204,  3401,  2491,
        11336,  1012,   102,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0])

In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained ("ProsusAI/finbert")
print (tokenizer.decode(batch["input_ids"][0]))

[CLS] the address location is provided to a local controller, preferably by wireless transmission, which then uses the address location to access the appliance control module. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


### Summary
The tokenized dataset has been successfully prepared for training.

key steps:
- Renamed `target` to `labels`
- Removed unnecessary columns (`sentence` and `label`)
- Converted features to PyTorch tensors
- Build DataLoaders for train, validation and test sets